# InternVL2-26B 적용 버전

베이스라인(Qwen2.5-VL-3B)에서 InternVL2-26B로 교체한 버전입니다.

**변경 사항 요약:**
- 모델: `Qwen2.5-VL-3B` → `OpenGVLab/InternVL2-26B`
- 모델 클래스: `AutoModelForImageTextToText` → `AutoModel`
- Processor: `AutoProcessor` → `AutoTokenizer` + 직접 이미지 전처리
- 이미지 해상도: 동적 448×448 타일 분할 방식
- LoRA rank: 16 유지
- 학습률: 2e-5 유지
- Epoch: 3 유지
- compute_dtype: bfloat16 (A100 최적화)

A100 GPU (40GB) 환경 / Colab Pro에서 실행하세요.

# 환경 준비

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade
!pip -q install einops timm  # InternVL2 필수 의존성

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# 압축 해제
!unzip "/content/2026_15_2_ai_DataSet.zip" -d "/content/"

In [ ]:
!ls /content/
!find /content -name "train.csv"
!find /content -name "test.csv"

# 라이브러리, 데이터, 설정

In [ ]:
import os, math, random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Any, List
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID = "OpenGVLab/InternVL2-26B"
IMG_SIZE = 448          # InternVL2 기본 타일 크기
MAX_TILES = 6           # 최대 타일 수 (A100 40GB 기준 안전값)
MAX_NEW_TOKENS = 8
EPOCHS = 3
SEED   = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

train_df = train_df.reset_index(drop=True)
print(f"전체 학습 데이터: {len(train_df)}개")
print(f"테스트 데이터:    {len(test_df)}개")

# 이미지 전처리

InternVL2는 AutoProcessor 대신 직접 이미지를 448×448 타일로 분할합니다.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=IMG_SIZE):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB")),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_best_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=MAX_TILES, image_size=IMG_SIZE, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1]
    )
    target_aspect_ratio = find_best_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized = image.resize((target_width, target_height))
    tiles = []
    for i in range(blocks):
        row = i // target_aspect_ratio[0]
        col = i % target_aspect_ratio[0]
        box = (col * image_size, row * image_size, (col + 1) * image_size, (row + 1) * image_size)
        tiles.append(resized.crop(box))
    if use_thumbnail and len(tiles) != 1:
        thumbnail = image.resize((image_size, image_size))
        tiles.append(thumbnail)
    return tiles

def load_image(image_path_or_pil, max_num=MAX_TILES):
    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert("RGB")
    else:
        image = image_path_or_pil.convert("RGB")
    transform = build_transform()
    tiles = dynamic_preprocess(image, max_num=max_num, use_thumbnail=True)
    pixel_values = torch.stack([transform(tile) for tile in tiles])
    return pixel_values

print("이미지 전처리 함수 정의 완료")

# 모델, Tokenizer

26B 모델 다운로드: 약 50GB, 20~30분 소요됩니다.

In [ ]:
# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
tokenizer.padding_side = "right"

# 모델 로드
base_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 설정 — 어텐션 레이어만 적용
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_internvl_prompt(user_text, num_tiles):
    """InternVL2 형식: <image> 태그로 이미지 위치 지정"""
    image_token = "<image>" * num_tiles
    return (
        f"<|im_start|>system\n{SYSTEM_INSTRUCT}<|im_end|>\n"
        f"<|im_start|>user\n{image_token}\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

# Custom Dataset, Collator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, tokenizer, train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        pixel_values = load_image(row["path"])
        num_tiles = pixel_values.shape[0]

        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )
        prompt = build_internvl_prompt(user_text, num_tiles)

        if self.train:
            gold = str(row["answer"]).strip().lower()
            full_text = prompt + gold + tokenizer.eos_token
        else:
            full_text = prompt

        return {
            "pixel_values": pixel_values,
            "text": full_text,
            "prompt_len": len(prompt) if self.train else None,
        }


@dataclass
class DataCollator:
    tokenizer: Any
    train: bool = True

    def __call__(self, batch):
        texts = [s["text"] for s in batch]
        pixel_values_list = [s["pixel_values"] for s in batch]

        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=2048,
            return_tensors="pt",
        )

        # 이미지 패딩: 타일 수가 다를 경우 최대 타일 수에 맞춰 패딩
        max_tiles = max(pv.shape[0] for pv in pixel_values_list)
        padded = []
        for pv in pixel_values_list:
            pad_size = max_tiles - pv.shape[0]
            if pad_size > 0:
                pv = torch.cat([pv, torch.zeros(pad_size, *pv.shape[1:])], dim=0)
            padded.append(pv)
        enc["pixel_values"] = torch.stack(padded)

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc

# DataLoader

In [ ]:
split = int(len(train_df) * 0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, tokenizer, train=True)
valid_ds = VQAMCDataset(valid_subset, tokenizer, train=True)

# 26B + 448px 타일 분할 → batch_size=1 필수
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=DataCollator(tokenizer, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                          collate_fn=DataCollator(tokenizer, True), num_workers=0)

print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

# Fine-tuning

In [ ]:
from tqdm.auto import tqdm

model_device = next(model.parameters()).device
GRAD_ACCUM = 8

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(num_training_steps * 0.03), num_training_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=False)

global_step = 0
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(model_device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(model_device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss: {val_loss / val_steps:.4f}")

# 모델 저장
SAVE_DIR = "/content/internvl2_26b_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)

# Inference

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"


torch.cuda.empty_cache()
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]

    pixel_values = load_image(row["path"]).to(model_device, dtype=torch.bfloat16)
    num_tiles = pixel_values.shape[0]

    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
    prompt = build_internvl_prompt(user_text, num_tiles)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    inputs["pixel_values"] = pixel_values.unsqueeze(0)  # [1, tiles, C, H, W]

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 프롬프트 부분 제거 후 응답만 디코딩
    generated = out_ids[:, inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    preds.append(extract_choice(output_text))

    del inputs, out_ids, pixel_values
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")
print(submission.head())

In [ ]:
# 마지막 추론 결과 확인
print(output_text)